In [ ]:

# -- Cell 1 -- rclone + Drive.
# Settings -> Internet ON, Accelerator GPU T4 x2, RCLONE_DRIVE_TOKEN attached.
# The second Kaggle account needs the same secret added under that account.
import os, subprocess

r = subprocess.run("curl -s https://rclone.org/install.sh | sudo bash", shell=True)
if r.returncode not in (0, 3):
    raise RuntimeError("rclone install failed (exit %d)" % r.returncode)

from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("RCLONE_DRIVE_TOKEN")
os.makedirs("/root/.config/rclone", exist_ok=True)
with open("/root/.config/rclone/rclone.conf", "w") as f:
    f.write("[drive]\ntype = drive\nscope = drive\ntoken = " + token + "\n")

REMOTE = "drive:Distillation"
out = subprocess.run("rclone lsf " + REMOTE, shell=True, capture_output=True, text=True)
print(out.stdout or out.stderr)
assert out.returncode == 0, "cannot see " + REMOTE


In [ ]:

# -- Cell 2 -- bisel8 sweep: PepMSND and AmpHGT, GPU-pinned, split across accounts.
#
# ONE notebook, two configurations. Edit PLAN in Cell 5 only.
#
#   NOTEBOOK 1    gpu0: PepMSND seed 101        gpu1: PepMSND seed 202
#   NOTEBOOK 2    gpu0: PepMSND seed 303        gpu1: AmpHGT seeds 202, 303
#
# Jobs are PINNED to a GPU and run sequentially within it, rather than being
# work-stolen by whichever card frees up. That is what you asked for and it is the
# right call for notebook 2, where one card runs a long fold sequence and the
# other runs two independent AmpHGT jobs. It does mean a card that finishes early
# idles instead of helping, so keep the two lists roughly balanced.
#
# TIMING. PepMSND is max_epochs 120 with patience 20 on val_mcc_best -- but that
# metric is MCC on 64 molecules, where one flipped molecule moves it ~0.03, so a
# clean 20-epoch plateau is unlikely and folds may well run all 120 epochs.
#
#     per PepMSND fold   66 min worst case (all 120 epochs)
#                        33-44 min if early stopping lands around epoch 60-80
#     10 folds           5.5 h best / 11.0 h worst   <- one GPU, one seed
#     per AmpHGT seed    ~170 min (measured, 8 blocks, batch 32)
#
# So each notebook is 5.5-11 h on its slowest card, against a 12 h session limit.
# EVERY job syncs to Drive the moment it finishes and is skipped on restart, so a
# session that dies part-way resumes rather than repeating work.
subprocess.run('pip install -q -U "transformers>=5.0" lightning peft', shell=True, check=True)
subprocess.run("pip uninstall -y -q torchao", shell=True)

import torch, numpy as np, pandas as pd, glob, json, time, shutil, threading
print("torch", torch.__version__, "| GPUs", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print("   cuda:%d %s %.0f GB" % (i, p.name, p.total_memory / 1e9))
assert torch.cuda.device_count() >= 2, "this plan pins jobs to cuda:0 and cuda:1"


In [ ]:

# -- Cell 3 -- their code and data, our code, both released models.
WORK = "/kaggle/working"
CODE, REPO = WORK + "/distill", WORK + "/project/their_repo"
TEACH = WORK + "/models/peptideclm-2-mlm-large"
SMALL = WORK + "/models/peptideclm-2-mlm-small"
os.makedirs(REPO, exist_ok=True)

def rlsf(path):
    r = subprocess.run("rclone lsf " + path, shell=True, capture_output=True, text=True)
    return r.stdout.split() if r.returncode == 0 else []

def pull_model(name, local):
    if os.path.exists(local + "/model.safetensors"):
        return "already local"
    flat = "%s/models/%s" % (REMOTE, name)
    if any(f.startswith("model.safetensors") for f in rlsf(flat)):
        os.makedirs(local, exist_ok=True)
        subprocess.run("rclone copy %s %s -P" % (flat, local), shell=True, check=True)
        return "flat"
    snaps = "%s/models/models--aaronfeller--%s/snapshots" % (REMOTE, name)
    shas = [x.rstrip("/") for x in rlsf(snaps)]
    assert shas, "%s not found. Tried  %s  and  %s" % (name, flat, snaps)
    os.makedirs(local, exist_ok=True)
    subprocess.run("rclone copy %s/%s %s -P" % (snaps, shas[0], local), shell=True, check=True)
    return "snapshot " + shas[0][:12]

for sub in ("data", "training"):
    if not os.path.isdir(REPO + "/" + sub):
        subprocess.run("rclone copy %s/their_repo/%s %s/%s --transfers 16 -P"
                       % (REMOTE, sub, REPO, sub), shell=True, check=True)
subprocess.run("rclone copy %s/distill %s --transfers 8 -P" % (REMOTE, CODE),
               shell=True, check=True)
print("teacher <- %s" % pull_model("peptideclm-2-mlm-large", TEACH))
print("small   <- %s" % pull_model("peptideclm-2-mlm-small", SMALL))

PEP_SCRIPTS = REPO + "/training/03_PepMSND_training_code/scripts"
PEP_PY = PEP_SCRIPTS + "/train_pepmsnd_kan_paperstyle.py"
PEP_DATA = REPO + "/data/PepMSND_data"
CLS_PY = REPO + "/training/02_classification_benchmarks_training_code/scripts/classification_finetuning_v2.py"
CLS_DATA = REPO + "/data"

if not os.path.exists(WORK + "/their_repo"):
    os.symlink(REPO, WORK + "/their_repo")
for name, src in (("peptideclm-2-mlm-large", TEACH), ("peptideclm-2-mlm-small", SMALL)):
    d = "%s/models/models--aaronfeller--%s/snapshots/local" % (WORK, name)
    if not os.path.exists(d):
        os.makedirs(os.path.dirname(d), exist_ok=True)
        os.symlink(src, d)

for p in (PEP_PY, PEP_SCRIPTS + "/kan.py", PEP_DATA + "/X_train1.csv",
          CLS_PY, CLS_DATA + "/amp_train.csv", CODE + "/export_truncated.py"):
    assert os.path.exists(p), "missing: " + p
print("ok -- both pipelines present")


In [ ]:

# -- Cell 4 -- bisel8, pulled from Drive so both accounts run identical weights.
EXPORT = WORK + "/compressed"
os.makedirs(EXPORT, exist_ok=True)
KEEP = "0,1,2,3,5,6,10,16"
ARM = EXPORT + "/peptideclm-2-mlm-bisel8"

if not os.path.exists(ARM + "/model.safetensors"):
    os.makedirs(ARM, exist_ok=True)
    subprocess.run("rclone copy %s/models/peptideclm-2-mlm-bisel8 %s -P" % (REMOTE, ARM),
                   shell=True, check=False)
if os.path.exists(ARM + "/model.safetensors"):
    print("bisel8 <- Drive")
else:
    print("not on Drive, deriving from the teacher")
    r = subprocess.run(["python", "export_truncated.py", "--out", ARM, "--keep", KEEP],
                       cwd=CODE, capture_output=True, text=True)
    print(r.stdout[-600:])
    assert r.returncode == 0, r.stderr[-1200:]

KEEP_LIST = [0, 1, 2, 3, 5, 6, 10, 16]
cfg = json.load(open(ARM + "/config.json"))
kept = cfg.get("pruned_from", {}).get("kept")
assert kept == KEEP_LIST, "wrong blocks in config: %s" % kept
# VERIFY THE WEIGHTS, NOT JUST THE CONFIG. A correct config.json sitting next to
# the wrong model.safetensors already cost this project four mislabelled
# benchmark runs. verify_weights compares exported block j against TEACHER block
# keep[j], tensor by tensor, so a swapped checkpoint cannot survive it.
_v = subprocess.run(["python", "-c",
                     "import sys; sys.path.insert(0, '.');"
                     "from export_truncated import verify_weights;"
                     "verify_weights(%r, %r, %r)" % (TEACH, ARM, KEEP_LIST)],
                    cwd=CODE, capture_output=True, text=True)
print(_v.stdout.strip() or _v.stderr[-800:])
assert _v.returncode == 0, (
    "THESE ARE NOT THE bisel8 WEIGHTS -- re-upload "
    "models/peptideclm-2-mlm-bisel8 to Drive.")

print("bisel8: %d blocks, %.1f MB, kept %s"
      % (cfg["num_blocks"], os.path.getsize(ARM + "/model.safetensors") / 1e6, kept))


In [ ]:

# -- Cell 5 -- THE PLAN. This is the only cell that differs between the notebooks.
#
# ============================================================================
#  NOTEBOOK 1 of 2 -- run on account A, simultaneously with notebook 2.
#    gpu0: PepMSND seed 101, folds 1-10
#    gpu1: PepMSND seed 202, folds 1-10
#  Nothing to edit. Notebook 2 covers seed 303 and AmpHGT.
PLAN = {0: [("pepmsnd", 101, f) for f in range(1, 11)],
        1: [("pepmsnd", 202, f) for f in range(1, 11)]}
# ============================================================================
OUT = WORK + "/results/bisel8_sweep"
os.makedirs(OUT, exist_ok=True)

# PepMSND hyperparameters, passed explicitly. Their script picks learning rates by
# SUFFIX MATCH on the model name: endswith("_lg") -> 7e-6 / 7e-4 / freeze 2,
# otherwise a fallback of 1e-5 / 1e-3 / freeze 3. Their shipped results used
# "aaronfeller_PeptideMLM_lg", so they got the _lg branch; our folder name matches
# nothing and would silently take the fallback -- a different optimiser setting
# from the 0.6180 we are comparing against.
PEP_LR, PEP_HEAD_LR, PEP_FREEZE = "7e-6", "7e-4", "2"
AMP_BS = "32"          # their published setting; 8 blocks fit at 32 on a T4

def job_dir(kind, seed, fold):
    return ("%s/pepmsnd/seed_%d/fold_%d" % (OUT, seed, fold) if kind == "pepmsnd"
            else "%s/amphgt/seed_%d" % (OUT, seed))

def job_done(kind, seed, fold):
    d = job_dir(kind, seed, fold)
    return bool(glob.glob(d + "/preds_fold%d.csv" % fold) if kind == "pepmsnd"
                else glob.glob(d + "/*_results.csv"))

def build_cmd(kind, seed, fold, d):
    if kind == "pepmsnd":
        return (["python", PEP_PY, "--model_name", ARM, "--fold", str(fold),
                 "--data_dir", PEP_DATA, "--save_path", d, "--seed", str(seed),
                 "--batch_size", "32",
                 "--backbone_learning_rate", PEP_LR,
                 "--head_learning_rate", PEP_HEAD_LR,
                 "--freeze_backbone_epochs", PEP_FREEZE], PEP_SCRIPTS)
    # their Trainer does devices=[int(args.gpu_index)] on the raw arg, default None
    return (["python", CLS_PY, "--dataset", "AmpHGT", "--gpu", "0", "--gpu_index", "0",
             "--model_name", ARM, "--batch_size", AMP_BS, "--seed", str(seed),
             "--data_dir", CLS_DATA, "--save_path", d,
             "--log_dir", "/tmp/logs/amp_%d" % seed], os.path.dirname(CLS_PY))

subprocess.run("rclone copy %s/results/bisel8_sweep %s --transfers 8 -P" % (REMOTE, OUT),
               shell=True, check=False)
for gpu, jobs in PLAN.items():
    pend = [j for j in jobs if not job_done(*j)]
    print("gpu%d: %d jobs, %d pending" % (gpu, len(jobs), len(pend)))


In [ ]:

# -- Cell 6 -- run each GPU's queue sequentially, the queues in parallel.
#
# One thread per GPU. Each finished job syncs to Drive immediately and its
# Lightning checkpoint is deleted -- 30 PepMSND checkpoints at 339 MB would be
# 10.2 GB against a 20 GB quota, and the predictions are the only output anything
# downstream reads.
t0 = time.time()
state = {g: "starting" for g in PLAN}

def run_queue(gpu, jobs):
    for kind, seed, fold in jobs:
        tag = "%s s%d%s" % (kind, seed, "" if fold is None else " f%d" % fold)
        if job_done(kind, seed, fold):
            print("[%5.1f min] gpu%d skip %s (already done)" % ((time.time()-t0)/60, gpu, tag))
            continue
        d = job_dir(kind, seed, fold)
        os.makedirs(d, exist_ok=True)
        cmd, cwd = build_cmd(kind, seed, fold, d)
        state[gpu] = tag
        print("[%5.1f min] gpu%d START %s" % ((time.time()-t0)/60, gpu, tag))
        p = subprocess.Popen(cmd, cwd=cwd, stdout=open(d + "/train.log", "w"),
                             stderr=subprocess.STDOUT,
                             env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu)))
        p.wait()
        ok = p.returncode == 0 and job_done(kind, seed, fold)
        print("[%5.1f min] gpu%d %s -> %s"
              % ((time.time()-t0)/60, gpu, tag, "ok" if ok else "FAILED rc=%s" % p.returncode))
        if ok:
            shutil.rmtree(os.path.join(d, "checkpoints"), ignore_errors=True)
            rel = d.replace(OUT, "").lstrip("/")
            subprocess.run("rclone copy %s %s/results/bisel8_sweep/%s "
                           "--exclude 'checkpoints/**' --drive-chunk-size 64M"
                           % (d, REMOTE, rel), shell=True, check=False)
        else:
            print("".join(open(d + "/train.log").readlines()[-15:]))
    state[gpu] = "done"

threads = [threading.Thread(target=run_queue, args=(g, j), daemon=True)
           for g, j in PLAN.items()]
for t in threads:
    t.start()

# Heartbeat. These jobs take 30-70 min each; without this a working run is
# indistinguishable from a hung one for over an hour.
while any(t.is_alive() for t in threads):
    time.sleep(600)
    print("   [%5.1f min] %s" % ((time.time()-t0)/60,
          " | ".join("gpu%d: %s" % (g, state[g]) for g in sorted(state))))
for t in threads:
    t.join()
print("")
print("all queues finished in %.2f h" % ((time.time() - t0) / 3600))


In [ ]:

# -- Cell 7 -- pool everything on Drive, from both accounts.
from sklearn.metrics import matthews_corrcoef, roc_auc_score, accuracy_score

subprocess.run("rclone copy %s/results/bisel8_sweep %s --transfers 8 -P" % (REMOTE, OUT),
               shell=True, check=False)

print("=== PepMSND (10 folds pooled per seed, threshold 0.5 on probabilities) ===")
rows = []
for sd in sorted(glob.glob(OUT + "/pepmsnd/seed_*")):
    s = int(os.path.basename(sd).split("_")[1])
    fs = sorted(glob.glob(sd + "/fold_*/preds_fold*.csv"))
    if not fs:
        continue
    d = pd.concat([pd.read_csv(f) for f in fs], ignore_index=True)
    y, p = d.true_label.values.astype(int), d.predicted_prob.values
    rows.append(dict(seed=s, folds=len(fs), n=len(d),
                     mcc=round(matthews_corrcoef(y, (p >= 0.5).astype(int)), 4),
                     auc=round(roc_auc_score(y, p), 4)))
if rows:
    r = pd.DataFrame(rows)
    print(r.to_string(index=False))
    full = r[r.folds == 10]
    if len(full):
        print("   bisel8 84.8M: MCC %.4f +- %.4f over %d complete seeds"
              % (full.mcc.mean(), full.mcc.std(), len(full)))
print("   their PeptideMTR_lg 337M      0.6613 +- 0.016")
print("   their PeptideMLM-MTR_lg 337M  0.6424 +- 0.022")
print("   their PeptideMLM_lg 337M      0.6180 +- 0.003   <- our teacher")
print("   descriptors alone             0.4871 | bag-of-tokens 0.4221")

print("\n=== AmpHGT (single split, threshold 0 on logits) ===")
rows = []
for sd in sorted(glob.glob(OUT + "/amphgt/seed_*")):
    s = int(os.path.basename(sd).split("_")[1])
    f = glob.glob(sd + "/*_results.csv")
    if not f:
        continue
    d = pd.read_csv(f[0])
    y, p = d.true_label.values.astype(int), d.predicted_label.values
    rows.append(dict(seed=s, n=len(d),
                     mcc=round(matthews_corrcoef(y, (p > 0).astype(int)), 4),
                     auc=round(roc_auc_score(y, p), 4),
                     acc=round(accuracy_score(y, (p > 0).astype(int)), 4)))
if rows:
    r = pd.DataFrame(rows)
    print(r.to_string(index=False))
    print("   (seed 101 measured earlier: MCC 0.8511)")
    allm = list(r.mcc) + [0.8511]
    print("   bisel8 84.8M across %d seeds: %.4f +- %.4f"
          % (len(allm), np.mean(allm), np.std(allm)))
print("   their mlm-large 337M   0.8844 +- 0.013")
print("   their mtr-large 337M   0.8531 | hybrid-large 0.8497")
print("   their xgboost-morgan   0.8356 | our warmstart32M 0.7921")
print("   bag-of-tokens control  0.6862")

subprocess.run("rclone copy %s %s/results/bisel8_sweep --exclude 'checkpoints/**' "
               "--drive-chunk-size 64M -P" % (OUT, REMOTE), shell=True, check=True)
print("\nuploaded -> %s/results/bisel8_sweep" % REMOTE)
